# Run experiments on Kaggle (GPU T4)

Runs the experiments of README §7 with the `pianoabs` package. Every step is **idempotent**: outputs that already exist are skipped, so after an interrupted session you can simply run all cells again.

Settings: *Accelerator* GPU T4, *Internet* on. Everything is written under `/kaggle/working/piano` (the repository clone), so it is kept in the notebook output.

Part 1 runs ablation A3 (λ_sp sweep) on **validation only** and stops. λ_sp for the `full` model is then fixed from the validation result (decision B5), committed, and Part 2 runs the test experiments.

In [ ]:
# 1. Code: clone the repository (or pull if it is already there)
import os, subprocess
REPO_URL = 'https://github.com/AllyPham04/piano.git'
WORK = '/kaggle/working/piano'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', REPO_URL, WORK], check=True)
else:
    subprocess.run(['git', '-C', WORK, 'pull', '--ff-only'], check=True)
os.chdir(WORK)
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

In [ ]:
# 2. System packages: FluidSynth + FluidR3_GM soundfont; check the soundfont path (README §5.5)
!apt-get -qq update && apt-get -qq install -y fluidsynth fluid-soundfont-gm > /dev/null
!fluidsynth --version | head -1
!find / -name 'FluidR3*.sf2' 2>/dev/null

In [ ]:
# 3. Python packages (torch, librosa, soundfile are preinstalled on Kaggle)
!pip install -q -e .
!python -m pianoabs.cli info --config configs/default.yaml

In [ ]:
# 4. Data: download MAESTRO MIDI, VERIFY the frozen manifests, build GT, render C1
!python -m pianoabs.cli prepare-data --config configs/default.yaml --manifests val test --conditions C1

In [ ]:
import subprocess, sys

def cli(*args):
    """Run one pianoabs CLI command and stream its output; stop on error."""
    cmd = [sys.executable, '-m', 'pianoabs.cli', *args, '--config', 'configs/default.yaml']
    print('$', ' '.join(cmd[2:]))
    subprocess.run(cmd, check=True)

def cell(method, condition, variant='full', test=True):
    """Tune thresholds on validation (C3 borrows C1), then run on the test manifest."""
    if condition != 'C3':
        cli('tune', '--method', method, '--variant', variant, '--condition', condition)
    if test:
        cli('run', '--method', method, '--variant', variant, '--condition', condition,
            '--manifest', 'test')

## Part 1: A3 on validation only (choose λ_sp)

In [ ]:
import json
for v in ['lsp_0', 'lsp_1e-3', 'lsp_1e-2', 'lsp_1e-1']:
    cell('ours', 'C1', v, test=False)
for v in ['lsp_0', 'lsp_1e-3', 'lsp_1e-2', 'lsp_1e-1']:
    r = json.load(open(f'results/thresholds/ours__{v}__C1.json'))
    print(f"{v:10s} val mean onset F1 = {r['val_mean_onset_F1']:.4f}")

**Stop here.** Set `fit.lambda_sp` in `configs/default.yaml` to the best validation value, commit, then re-run cell 1 (pull) and continue with Part 2.

## Part 2: test experiments

In [ ]:
# E0 (C0 sanity check) and E1 (C1) for the label-free methods
cell('ours', 'C0')
cell('ours', 'C1')
cell('nmf_harmonic', 'C1')

In [ ]:
# Ablations on C1: A1 (B = 0), A2 (decay fixed), A3 (lambda_sp on test), A4 (single resolution)
for v in ['B0', 'rho_fixed', 'single_res', 'lsp_0', 'lsp_1e-3', 'lsp_1e-2', 'lsp_1e-1']:
    cell('ours', 'C1', v)

In [ ]:
# Metrics, bootstrap CIs, Wilcoxon tests -> results/*.csv
cli('evaluate', '--manifest', 'test')

In [ ]:
# Package results for download (small files only; audio stays out)
!cd /kaggle/working && zip -qr results_bundle.zip piano/results piano/outputs/predictions piano/outputs/params && ls -la results_bundle.zip